# 02D — Condition_raw semantic layer


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 top conditions (cleaned)


In [3]:
top_cond = d['condition_main'].value_counts().head(20).reset_index()
top_cond.columns = ['condition', 'count']
display(top_cond)
fig = px.bar(top_cond, x='condition', y='count', title='Top cleaned conditions')
fig.update_xaxes(tickangle=-45)
fig.show()


,condition,count
0,duchenne muscular dystrophy,6793
1,cardiovascular phenotype,752
2,becker muscular dystrophy cardiomyopathy duche...,560
3,becker muscular dystrophy,199
4,dilated cardiomyopathy 3b,162
5,neuromuscular disease caused by qualitative or...,161
6,dmd-related disorder,27
7,elevated circulating creatine kinase concentra...,14
8,abnormality of the musculature,13
9,progressive muscular dystrophy,7


## 2. 📊 condition → phenotype mapping


In [4]:
top = d['condition_main'].value_counts().head(20).index
tmp = d[d['condition_main'].isin(top)]
map_tbl = tmp.groupby(['condition_main', 'phenotype']).size().reset_index(name='n')
pivot = map_tbl.pivot(index='condition_main', columns='phenotype', values='n').fillna(0)
pivot['total'] = pivot.sum(axis=1)
pivot['dominant_phenotype'] = pivot.drop(columns='total').idxmax(axis=1)
pivot['dominant_fraction'] = (pivot.drop(columns=['total', 'dominant_phenotype']).max(axis=1) / pivot['total']).round(3)
display(pivot.sort_values('total', ascending=False).head(20))


phenotype,BMD,DMD,total,dominant_phenotype,dominant_fraction
condition_main,,,,,
duchenne muscular dystrophy,135.0,6658.0,6793.0,DMD,0.980
becker muscular dystrophy cardiomyopathy duchenne muscular dystrophy dystrophin deficiency,560.0,0.0,560.0,BMD,1.000
cardiovascular phenotype,38.0,431.0,469.0,DMD,0.919
becker muscular dystrophy,199.0,0.0,199.0,BMD,1.000
dilated cardiomyopathy 3b,41.0,45.0,86.0,DMD,0.523
neuromuscular disease caused by qualitative or quantitative defects of dystrophin,3.0,12.0,15.0,DMD,0.800
dmd-related disorder,1.0,6.0,7.0,DMD,0.857
primary dilated cardiomyopathy,0.0,4.0,4.0,DMD,1.000
progressive muscular dystrophy,0.0,4.0,4.0,DMD,1.000


## 3. 🧪 χ² condition ~ phenotype


In [5]:
tmp = d[d['condition_main'].isin(d['condition_main'].value_counts()[lambda s: s >= 20].index)]
tab, chi2, p, dof = chi2_table(tmp, 'condition_main', 'phenotype')
display(tab.head(30))
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


phenotype,BMD,DMD
condition_main,,
becker muscular dystrophy,199,0
becker muscular dystrophy cardiomyopathy duchenne muscular dystrophy dystrophin deficiency,560,0
cardiovascular phenotype,38,431
dilated cardiomyopathy 3b,41,45
dmd-related disorder,1,6
duchenne muscular dystrophy,135,6658
neuromuscular disease caused by qualitative or quantitative defects of dystrophin,3,12


,value
chi2,6313.744968
dof,6.000000
p_value,0.000000


## 4. 📊 condition vs mutation_type


In [6]:
top = d['condition_main'].value_counts().head(15).index
tmp = d[d['condition_main'].isin(top)][['condition_main', 'mutation_type']]
fig = px.histogram(tmp, x='condition_main', color='mutation_type', barmode='stack', title='Condition vs mutation_type')
fig.update_xaxes(tickangle=-45)
fig.show()


## 5. 🧪 χ² condition ~ mutation


In [7]:
tmp = d[d['condition_main'].isin(d['condition_main'].value_counts()[lambda s: s >= 20].index)]
tab, chi2, p, dof = chi2_table(tmp, 'condition_main', 'mutation_type')
display(tab.head(30))
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


mutation_type,frameshift,large_deletion,missense,nonsense,splice
condition_main,,,,,
becker muscular dystrophy,38,24,32,44,34
becker muscular dystrophy cardiomyopathy duchenne muscular dystrophy dystrophin deficiency,2,0,274,0,6
cardiovascular phenotype,3,12,411,30,38
dilated cardiomyopathy 3b,9,2,55,7,10
dmd-related disorder,1,1,9,3,2
duchenne muscular dystrophy,482,1052,1874,477,590
neuromuscular disease caused by qualitative or quantitative defects of dystrophin,11,121,3,11,10


,value
chi2,1.052805e+03
dof,2.400000e+01
p_value,5.354686e-207


## 6. 📊 condition vs pathogenic


In [8]:
top = d['condition_main'].value_counts().head(20).index
tbl = d[d['condition_main'].isin(top)].groupby('condition_main').agg(pathogenic_fraction=('pathogenic', 'mean'), n=('var_id', 'size')).reset_index().sort_values('n', ascending=False)
display(tbl)
fig = px.bar(tbl, x='condition_main', y='pathogenic_fraction', hover_data=['n'], title='Condition vs pathogenic fraction')
fig.update_xaxes(tickangle=-45)
fig.show()


,condition_main,pathogenic_fraction,n
8,duchenne muscular dystrophy,0.314736,6793
3,cardiovascular phenotype,0.027926,752
2,becker muscular dystrophy cardiomyopathy duche...,0.014286,560
1,becker muscular dystrophy,0.824121,199
5,dilated cardiomyopathy 3b,0.135802,162
13,neuromuscular disease caused by qualitative or...,0.987578,161
6,dmd-related disorder,0.148148,27
9,elevated circulating creatine kinase concentra...,0.928571,14
0,abnormality of the musculature,1.000000,13
17,progressive muscular dystrophy,0.428571,7


## 7. 🧪 Fisher condition ~ pathogenic


In [9]:
top = d['condition_main'].value_counts()[lambda s: s >= 30].index
rows = []
for cnd in top:
    tmp = d[['condition_main', 'pathogenic']].dropna().copy()
    tmp['is_condition'] = tmp['condition_main'].eq(cnd)
    tab, odds, p, ci = fisher_bool(tmp, 'is_condition', 'pathogenic')
    rows.append({'condition': cnd, 'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]})
res = pd.DataFrame(rows).sort_values('p_value')
display(res.head(30))


,condition,odds_ratio,p_value,or_ci_low,or_ci_high
1,cardiovascular phenotype,0.060872,1.814136e-87,0.039347,0.094170
5,neuromuscular disease caused by qualitative or...,201.911001,1.287521e-82,50.019740,815.039278
2,becker muscular dystrophy cardiomyopathy duche...,0.031560,1.615389e-74,0.015680,0.063524
3,becker muscular dystrophy,11.861319,3.927006e-55,8.209723,17.137106
0,duchenne muscular dystrophy,1.549082,8.512069e-14,1.377743,1.741729
4,dilated cardiomyopathy 3b,0.369322,2.061684e-06,0.235049,0.580302


## 8. 📊 condition vs exon hotspot ⭐


In [10]:
top = d['condition_main'].value_counts().head(20).index
tbl = d[d['condition_main'].isin(top)].groupby('condition_main').agg(hotspot_fraction=('is_hotspot_distal', 'mean'), n=('var_id', 'size')).reset_index().sort_values('n', ascending=False)
display(tbl)
fig = px.bar(tbl, x='condition_main', y='hotspot_fraction', hover_data=['n'], title='Condition vs distal hotspot fraction (exon 45-55)')
fig.update_xaxes(tickangle=-45)
fig.show()


,condition_main,hotspot_fraction,n
8,duchenne muscular dystrophy,0.155749,6793
3,cardiovascular phenotype,0.159574,752
2,becker muscular dystrophy cardiomyopathy duche...,0.164286,560
1,becker muscular dystrophy,0.150754,199
5,dilated cardiomyopathy 3b,0.092593,162
13,neuromuscular disease caused by qualitative or...,0.354037,161
6,dmd-related disorder,0.111111,27
9,elevated circulating creatine kinase concentra...,0.071429,14
0,abnormality of the musculature,0.076923,13
17,progressive muscular dystrophy,0.142857,7


## 9. 🧪 Fisher condition hotspot enrichment 📖


In [11]:
top = d['condition_main'].value_counts()[lambda s: s >= 30].index
rows = []
for cnd in top:
    tmp = d[['condition_main', 'is_hotspot_distal']].dropna().copy()
    tmp['is_condition'] = tmp['condition_main'].eq(cnd)
    tab, odds, p, ci = fisher_bool(tmp, 'is_condition', 'is_hotspot_distal')
    rows.append({'condition': cnd, 'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]})
res = pd.DataFrame(rows).sort_values('p_value')
display(res.head(30))


,condition,odds_ratio,p_value,or_ci_low,or_ci_high
5,neuromuscular disease caused by qualitative or...,3.000174,1.008882e-09,2.160639,4.165918
4,dilated cardiomyopathy 3b,0.538230,2.179105e-02,0.315415,0.918444
0,duchenne muscular dystrophy,0.924104,2.607632e-01,0.806771,1.058501
2,becker muscular dystrophy cardiomyopathy duche...,1.049853,6.753012e-01,0.833068,1.323051
3,becker muscular dystrophy,0.943818,8.444201e-01,0.637342,1.397667
1,cardiovascular phenotype,1.011857,9.167387e-01,0.824959,1.241098


## 10. 📊 noisy condition detection (long tail)


In [12]:
counts = d['condition_main'].value_counts()
rank_tbl = counts.reset_index()
rank_tbl.columns = ['condition', 'count']
rank_tbl['rank'] = np.arange(1, len(rank_tbl) + 1)
fig = px.line(rank_tbl, x='rank', y='count', log_x=True, log_y=True, title='Condition long-tail rank-frequency')
fig.show()


## 11. 📋 filtered vs raw condition table


In [13]:
tmp = d[['condition_raw', 'condition_main']].copy()
raw_counts = tmp['condition_raw'].value_counts(dropna=False).head(30).reset_index()
raw_counts.columns = ['condition_raw', 'count']
map_tbl = tmp.groupby(['condition_raw', 'condition_main']).size().reset_index(name='n').sort_values('n', ascending=False)
display(raw_counts)
display(map_tbl.head(40))


,condition_raw,count
0,Duchenne muscular dystrophy,5841
1,not provided,1216
2,"Becker muscular dystrophy, Cardiomyopathy, Duc...",559
3,not provided|Duchenne muscular dystrophy,340
4,Duchenne muscular dystrophy|not provided,291
5,Duchenne muscular dystrophy|Cardiovascular phe...,281
6,Cardiovascular phenotype,274
7,NaN,254
8,Cardiovascular phenotype|Duchenne muscular dys...,239
9,See cases,185


,condition_raw,condition_main,n
167,Duchenne muscular dystrophy,duchenne muscular dystrophy,5841
7,"Becker muscular dystrophy, Cardiomyopathy, Duc...",becker muscular dystrophy cardiomyopathy duche...,559
245,Duchenne muscular dystrophy|not provided,duchenne muscular dystrophy,291
190,Duchenne muscular dystrophy|Cardiovascular phe...,duchenne muscular dystrophy,281
30,Cardiovascular phenotype,cardiovascular phenotype,274
46,Cardiovascular phenotype|Duchenne muscular dys...,cardiovascular phenotype,239
283,Neuromuscular disease caused by qualitative or...,neuromuscular disease caused by qualitative or...,143
6,Becker muscular dystrophy,becker muscular dystrophy,114
118,Dilated cardiomyopathy 3B,dilated cardiomyopathy 3b,66
199,Duchenne muscular dystrophy|Cardiovascular phe...,duchenne muscular dystrophy,57


## 12. 📊 condition entropy 📖


In [14]:
p = d['condition_main'].value_counts(normalize=True)
ent = entropy(p.values, base=2)
out = pd.Series({'n_conditions': int(p.shape[0]), 'entropy_bits': float(ent)})
display(out.to_frame('value'))


,value
n_conditions,44.000000
entropy_bits,1.328194
